In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 및 폰트 통합 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False     
    }
)

# ==========================================
# 2. 경로 및 환경 변수 세팅
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P3")
os.makedirs(SAVE_DIR, exist_ok=True)

MODELS = {
    "Llama-3.2-1B-Instruct": {"layers": 16},
    "Qwen2.5-1.5B-Instruct": {"layers": 28}
}

BIT_LEVELS = ["Original_BF16", "GPTQ_8bit", "GPTQ_4bit", "GPTQ_3bit", "GPTQ_2bit"]

# ==========================================
# 3. 데이터 로드 및 차분(Delta) 연산 유틸리티
# ==========================================
def load_layer_statistics(model_name, bit_suffix, prompt_dir):
    """특정 프롬프트(P1 또는 P3)의 layer_statistics.json을 읽어 DataFrame 반환 (방탄 로직 적용)"""
    folder_name = f"{model_name}_{bit_suffix}"
    json_path = os.path.join(BASE_DIR, folder_name, prompt_dir, "layer_statistics.json")
    
    if not os.path.exists(json_path):
        print(f"Warning: Data not found at {json_path}")
        return pd.DataFrame()
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    parsed_data = []
    
    # [방탄 파싱 로직]
    if 'layers' in data and isinstance(data['layers'], list):
        for item in data['layers']:
            if 'module_for_decoder_layer' in item:
                parsed_data.append({
                    "layer": int(item['module_for_decoder_layer']),
                    "attn_l2": item.get('attn_global_l2_norm', 0.0),
                    "mlp_l2": item.get('mlp_global_l2_norm', 0.0)
                })
    elif isinstance(data, dict):
        for key, layer_info in data.items():
            if str(key).isdigit() and isinstance(layer_info, dict): 
                parsed_data.append({
                    "layer": int(key),
                    "attn_l2": layer_info.get("attn_global_l2_norm", 0.0),
                    "mlp_l2": layer_info.get("mlp_global_l2_norm", 0.0)
                })

    if not parsed_data:
        return pd.DataFrame()
        
    return pd.DataFrame(parsed_data).sort_values(by="layer")

def get_delta_dataframe(model_name, bit_suffix):
    """P1 대비 P3의 차분(Delta) 데이터프레임을 생성합니다 (Delta = P3 - P1)"""
    df_p1 = load_layer_statistics(model_name, bit_suffix, "Prompt_01")
    df_p3 = load_layer_statistics(model_name, bit_suffix, "Prompt_03")
    
    if df_p1.empty or df_p3.empty:
        return pd.DataFrame()
        
    # Layer를 기준으로 병합
    df_merged = pd.merge(df_p1, df_p3, on="layer", suffixes=("_p1", "_p3"))
    
    # Delta (차분) 계산: 포맷 강제가 유발한 '추가적인 부하'
    df_merged["delta_attn"] = df_merged["attn_l2_p3"] - df_merged["attn_l2_p1"]
    df_merged["delta_mlp"] = df_merged["mlp_l2_p3"] - df_merged["mlp_l2_p1"]
    
    return df_merged

# ==========================================
# 4. 시각화 1: Model-wise Delta Attention Error Spike (3-bit)
# ==========================================
def plot_delta_attn_spike_3bit():
    plt.figure(figsize=(12, 6))
    colors = {"Llama-3.2-1B-Instruct": "blue", "Qwen2.5-1.5B-Instruct": "purple"}
    
    for model_name in MODELS.keys():
        df_delta = get_delta_dataframe(model_name, "GPTQ_3bit")
        if df_delta.empty: continue
            
        plt.plot(df_delta["layer"], df_delta["delta_attn"], 
                 label=f"{model_name}", color=colors[model_name], 
                 marker='o', linewidth=2.5)

    plt.title("[P3 vs P1] Formatting Tax: Delta Attention Error (3-bit)", fontsize=15, fontweight='bold')
    plt.xlabel("Layer Depth")
    plt.ylabel("$\Delta$ Attention L2 Error (P3 - P1)")
    plt.axhline(y=0, color='black', linestyle='--', linewidth=1) # 0 기준선
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig1_P3_Delta_Attn_Spike.png"), dpi=300)
    plt.close()

# ==========================================
# 5. 시각화 2: Qwen 3-bit 내부 Delta Attn vs Delta MLP 대조
# ==========================================
def plot_qwen_delta_comparison():
    target_model = "Qwen2.5-1.5B-Instruct"
    df_delta = get_delta_dataframe(target_model, "GPTQ_3bit")
    
    if df_delta.empty: return
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(df_delta["layer"]))
    width = 0.4
    
    # 막대 그래프 그리기
    ax.bar(x - width/2, df_delta["delta_attn"], width, label='$\Delta$ Attention Error (포맷 부하)', color='red', alpha=0.8)
    ax.bar(x + width/2, df_delta["delta_mlp"], width, label='$\Delta$ MLP Error (논리 부하)', color='blue', alpha=0.5)
    
    ax.set_title(f"[{target_model}] Formatting Tax Impact: Attn vs MLP (3-bit)", fontsize=15, fontweight='bold')
    ax.set_xlabel("Layer Depth")
    ax.set_ylabel("Error Increase ($\Delta$ P3 - P1)")
    ax.set_xticks(x)
    ax.set_xticklabels(df_delta["layer"])
    ax.legend()
    ax.grid(True, axis='y', linestyle="--", alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig2_P3_Qwen_Delta_Comparison.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 시각화 3: Qwen Bit-wise Delta Attention Heatmap
# ==========================================
def plot_qwen_bitwise_heatmap():
    target_model = "Qwen2.5-1.5B-Instruct"
    heatmap_data = []
    y_labels = []
    
    # 16-bit부터 3-bit까지만 역순으로 추출 (2-bit는 Dead 상태일 수 있으므로 제외 권장, 필요시 포함)
    target_bits = ["GPTQ_3bit", "GPTQ_4bit", "GPTQ_8bit", "Original_BF16"] 
    
    for bit in target_bits:
        df_delta = get_delta_dataframe(target_model, bit)
        if not df_delta.empty:
            heatmap_data.append(df_delta["delta_attn"].values)
            y_labels.append(bit.replace("Original_", "").replace("GPTQ_", ""))
            layers = df_delta["layer"].values # X축 라벨용
            
    if not heatmap_data: return

    plt.figure(figsize=(14, 4))
    sns.heatmap(heatmap_data, cmap="Reds", center=0,
                yticklabels=y_labels, xticklabels=layers,
                cbar_kws={'label': '$\Delta$ Attention Error'})
                
    plt.title(f"[{target_model}] Bit-wise Formatting Tax Accumulation (Heatmap)", fontsize=15, fontweight='bold')
    plt.xlabel("Layer Depth")
    plt.ylabel("Quantization Bit Level")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig3_P3_Qwen_Bitwise_Heatmap.png"), dpi=300)
    plt.close()

# ==========================================
# 7. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P3 포맷 강제성(Formatting Tax) 차분 분석 파이프라인 가동 중...")
    
    print(" 1/3. Cross-Model Delta Attention Spike 그래프 생성 중...")
    plot_delta_attn_spike_3bit()
    
    print(" 2/3. Qwen Delta Attn vs MLP 막대그래프 생성 중...")
    plot_qwen_delta_comparison()
    
    print(" 3/3. Qwen Bit-wise Heatmap 생성 중...")
    plot_qwen_bitwise_heatmap()
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 안전하게 저장되었습니다.")

P3 포맷 강제성(Formatting Tax) 차분 분석 파이프라인 가동 중...
 1/3. Cross-Model Delta Attention Spike 그래프 생성 중...
 2/3. Qwen Delta Attn vs MLP 막대그래프 생성 중...
 3/3. Qwen Bit-wise Heatmap 생성 중...

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P3'에 안전하게 저장되었습니다.


In [3]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False
    }
)

# ==========================================
# 2. 경로 및 분석 타겟 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P3_Activation")
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_MODEL = "Qwen2.5-1.5B-Instruct"
TARGET_BIT = "GPTQ_3bit"
TARGET_LAYER = 26
BLOCK_TYPE = "mlp"

# ==========================================
# 3. 텐서 로드 및 전처리 유틸리티
# ==========================================
def load_and_clean_tensor(prompt_dir):
    """텐서를 로드하고 NaN/Inf 등 시각화를 방해하는 오류 값을 제거하여 1차원 배열로 반환합니다."""
    folder_name = f"{TARGET_MODEL}_{TARGET_BIT}"
    tensor_name = f"layer_{TARGET_LAYER}_{BLOCK_TYPE}_output.pt"
    tensor_path = os.path.join(BASE_DIR, folder_name, prompt_dir, "tensors", tensor_name)
    
    if not os.path.exists(tensor_path):
        print(f"Warning: Tensor not found at {tensor_path}. Using mock data.")
        seq_len, dim = 200, 2048
        if prompt_dir == "Prompt_01":
            # P1 Mock: 넓은 분산과 소수의 극단적 이상치
            out = torch.randn(seq_len, dim) * 2.0
            out[:, torch.randint(0, dim, (5,))] = torch.randn(seq_len, 5) * 500.0
            np_arr = out.flatten().numpy()
        else:
            # P3 Mock: 거의 완벽하게 0으로 수렴한 Dead 상태
            out = torch.randn(seq_len, dim) * 0.0001
            np_arr = out.flatten().numpy()
    else:
        np_arr = torch.load(tensor_path)[0].flatten().numpy()
        
    # 유효한 숫자(Finite)만 필터링
    return np_arr[np.isfinite(np_arr)]

# ==========================================
# 4. 시각화 1: 전역 거시 뷰 (Log-Scale Histogram)
# ==========================================
def plot_macro_log_histogram(arr_p1, arr_p3):
    """Y축을 Log 스케일로 설정하여 극단적 이상치(P1)와 영점 밀집(P3)을 동시에 보여줍니다."""
    plt.figure(figsize=(12, 6))
    
    # 투명도를 주어 두 히스토그램을 겹쳐 그립니다.
    plt.hist(arr_p1, bins=150, log=True, alpha=0.5, color='purple', label="P1 (L2 Hallucination - 꼬리가 김)")
    plt.hist(arr_p3, bins=150, log=True, alpha=0.6, color='red', label="P3 (L1 Dead Activation - 0에 뭉침)")
    
    plt.title(f"[{TARGET_MODEL}] Layer {TARGET_LAYER} {BLOCK_TYPE.upper()} Macro View (Log Scale)", fontsize=14, fontweight='bold')
    plt.xlabel("Activation Value")
    plt.ylabel("Frequency (Log Scale)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4A_P3_Macro_Log_Hist.png"), dpi=300)
    plt.close()

# ==========================================
# 5. 시각화 2: 영점 미시 뷰 (Zoom-in Linear Histogram)
# ==========================================
def plot_micro_zero_histogram(arr_p1, arr_p3):
    """0.0 주변을 확대하고 선형 스케일을 사용하여 '바늘(Needle)'의 물리적 높이를 보여줍니다."""
    plt.figure(figsize=(10, 6))
    
    # -0.5 ~ 0.5 구간만 정밀 타격
    zoom_range = (-0.5, 0.5)
    
    plt.hist(arr_p1, bins=100, range=zoom_range, alpha=0.4, color='purple', label="P1 (분산 유지)")
    plt.hist(arr_p3, bins=100, range=zoom_range, alpha=0.7, color='red', label="P3 (0.0으로 수렴한 바늘)")
    
    plt.title(f"[{TARGET_MODEL}] Layer {TARGET_LAYER} {BLOCK_TYPE.upper()} Micro View (Zoomed near 0.0)", fontsize=14, fontweight='bold')
    plt.xlabel("Activation Value")
    plt.ylabel("Frequency (Linear Scale)")
    plt.xlim(-0.5, 0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4B_P3_Micro_Zero_Hist.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 시각화 3: Dead Activation 정량화 (Sparsity Bar Chart)
# ==========================================
def plot_dead_activation_quantification(arr_p1, arr_p3):
    """값이 0에 극도로 가까운(Dead) 텐서의 '비율(%)'을 수치적으로 확정 짓습니다."""
    # 임계치(Threshold): 0.001 미만이면 기능이 정지된(Dead) 뉴런으로 간주
    threshold = 1e-3
    
    dead_ratio_p1 = np.mean(np.abs(arr_p1) < threshold) * 100
    dead_ratio_p3 = np.mean(np.abs(arr_p3) < threshold) * 100
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(["P1 (순수 추론)", "P3 (포맷 강제)"], [dead_ratio_p1, dead_ratio_p3], color=['purple', 'red'], alpha=0.8)
    
    plt.title(f"[{TARGET_MODEL}] Percentage of 'Dead' Activations (< {threshold})", fontsize=14, fontweight='bold')
    plt.ylabel("Dead Activation Ratio (%)")
    plt.ylim(0, 105)
    
    # 막대 위에 수치 텍스트 표기
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 2,
                 f'{height:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
                 
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4C_P3_Dead_Ratio_Bar.png"), dpi=300)
    plt.close()

# ==========================================
# 7. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P3 심층 활성화 분포 3중 교차 분석을 가동합니다...")
    
    print(" -> 데이터 로드 및 전처리 중...")
    tensor_p1 = load_and_clean_tensor("Prompt_01")
    tensor_p3 = load_and_clean_tensor("Prompt_03")
    
    print(" 1/3. 거시 분포 (Log-Scale) 차트 생성 중...")
    plot_macro_log_histogram(tensor_p1, tensor_p3)
    
    print(" 2/3. 미시 영점 뷰 (Zoom-in) 차트 생성 중...")
    plot_micro_zero_histogram(tensor_p1, tensor_p3)
    
    print(" 3/3. Dead Activation 정량화 차트 생성 중...")
    plot_dead_activation_quantification(tensor_p1, tensor_p3)
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 저장되었습니다.")

P3 심층 활성화 분포 3중 교차 분석을 가동합니다...
 -> 데이터 로드 및 전처리 중...
 1/3. 거시 분포 (Log-Scale) 차트 생성 중...
 2/3. 미시 영점 뷰 (Zoom-in) 차트 생성 중...
 3/3. Dead Activation 정량화 차트 생성 중...

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P3_Activation'에 저장되었습니다.
